# 🛠️ Step 1: Data Engineering Pipeline
This notebook handles the transformation of raw Pascal VOC data into a high-performance YOLOv8 format. 

**Pipeline:** Raw VOC $\rightarrow$ Supervision Dataset $\rightarrow$ Split (80/10/10) $\rightarrow$ YOLO Format $\rightarrow$ `data.yaml`

In [ ]:
import os
import yaml
import shutil
import matplotlib.pyplot as plt
import cv2
import random
import supervision as sv
from pathlib import Path

BASE_DIR = Path(os.getcwd()).absolute()
IMG_DIR = (BASE_DIR / 'data' / 'images').absolute()
ANN_DIR = (BASE_DIR / 'data' / 'annotations').absolute()
FINAL_DIR = (BASE_DIR / 'HelmetDataset').absolute()

print(f"🚀 Working Directory: {BASE_DIR}")
print(f"📂 Dataset Source: {IMG_DIR}")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1. Load and Split Dataset
dataset = sv.DetectionDataset.from_pascal_voc(
    images_directory_path=str(IMG_DIR),
    annotations_directory_path=str(ANN_DIR)
)

train_ds, rem_ds = dataset.split(split_ratio=0.8)
val_ds, test_ds = rem_ds.split(split_ratio=0.5)

print(f"✅ Dataset Loaded. Splits -> Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# 2. Export to YOLO Format
splits = {'train': train_ds, 'val': val_ds, 'test': test_ds}
for name, ds in splits.items():
    ds.as_yolo(
        images_directory_path=str(FINAL_DIR / f"{name}/images"),
        annotations_directory_path=str(FINAL_DIR / f"{name}/labels")
    )

# 3. Generate Professional data.yaml
data_config = {
    'path': str(FINAL_DIR).replace('\', '/'),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(dataset.classes),
    'names': dataset.classes
}

with open(FINAL_DIR / 'data.yaml', 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print("✅ YOLO Dataset and data.yaml ready!")